# Data Cleaning and Outlier Detection

This notebook performs the core cleaning steps from the legacy exploratory pipeline. It produces a cleaned dataset and detects outliers for later analysis and feature engineering.

## Objectives

- Load the filtered raw dataset produced by the exploratory notebook.
- Remove incomplete, invalid, and non-conforming rows.
- Remove duplicate records and weak outliers.
- Save the cleaned dataset for feature engineering.
- Detect energy and surface outliers and save them for review.

In [ ]:
import os
import numpy as np
import pandas as pd

## 1. Load the filtered dataset

The exploratory notebook should produce `data/2016_Building_Energy_Benchmarking_Purge.csv`. This notebook continues from that result.

In [ ]:
purge_path = "data/2016_Building_Energy_Benchmarking_Purge.csv"
if not os.path.exists(purge_path):
    raise FileNotFoundError(f"Expected purge file not found: {purge_path}")

df = pd.read_csv(purge_path)
print("Loaded purge dataset:", purge_path)
print("Shape:", df.shape)
print("Columns:", len(df.columns))

## 2. Remove records with critical missing values

Keep only rows with the required core fields for energy modeling.

In [ ]:
critical_cols = [
    'SiteEnergyUse(kBtu)',
    'TotalGHGEmissions',
    'PropertyGFATotal',
    'PropertyGFABuilding(s)',
    'ComplianceStatus'
]
missing_mask = df[critical_cols].isnull().any(axis=1)
print("Critical missing rows:", missing_mask.sum())
df = df[~missing_mask].reset_index(drop=True)
print("Remaining rows after critical missing removal:", df.shape[0])

## 3. Remove non-conforming status records

Drop rows with poor or error compliance states.

In [ ]:
non_conform_values = ['Error - Correct Default Data', 'Non-Compliant']
mask_non_conform = df['ComplianceStatus'].isin(non_conform_values)
print("Non-conforming rows:", mask_non_conform.sum())
df = df[~mask_non_conform].reset_index(drop=True)
print("Remaining rows after compliance filtering:", df.shape[0])

## 4. Remove physically invalid values

Drop rows where key numeric values are zero or negative.

In [ ]:
invalid_mask = (
    (df['SiteEnergyUse(kBtu)'] <= 0) |
    (df['TotalGHGEmissions'] <= 0) |
    (df['PropertyGFATotal'] <= 0) |
    (df['PropertyGFABuilding(s)'] <= 0)
)
print("Physically invalid rows:", invalid_mask.sum())
df = df[~invalid_mask].reset_index(drop=True)
print("Remaining rows after invalid value removal:", df.shape[0])

## 5. Remove low outlier annotations and duplicate rows

If the dataset contains an `Outlier` column, remove rows marked as low outliers, then remove duplicates.

In [ ]:
if 'Outlier' in df.columns:
    low_outliers = df['Outlier'] == 'Low outlier'
    print("Low outlier rows:", low_outliers.sum())
    df = df[~low_outliers].reset_index(drop=True)
    df = df.drop(columns=['Outlier'], errors='ignore')
    print("Remaining rows after low outlier removal:", df.shape[0])
else:
    print("No 'Outlier' column found; skipping low outlier removal.")

duplicate_count = df.duplicated().sum()
print("Duplicate rows found:", duplicate_count)
if duplicate_count > 0:
    df = df.drop_duplicates().reset_index(drop=True)
    print("Remaining rows after duplicate removal:", df.shape[0])

## 6. Drop unhelpful or redundant columns

Remove columns that are non-predictive, too specific, or strongly redundant with the target.

In [ ]:
columns_to_drop = [
    'ListOfAllPropertyUseTypes',
    'SecondLargestPropertyUseType',
    'SecondLargestPropertyUseTypeGFA',
    'ThirdLargestPropertyUseType',
    'ThirdLargestPropertyUseTypeGFA',
    'YearsENERGYSTARCertified',
    'ENERGYSTARScore',
    'Neighborhood',
    'Latitude',
    'Longitude',
    'PropertyName',
    'Address',
    'City',
    'State',
    'ZipCode',
    'TaxParcelIdentificationNumber',
    'CouncilDistrictCode',
    'SiteEUI(kBtu/sf)',
    'SiteEUIWN(kBtu/sf)',
    'SourceEUI(kBtu/sf)',
    'SourceEUIWN(kBtu/sf)',
    'Electricity(kWh)',
    'Electricity(kBtu)',
    'NaturalGas(therms)',
    'NaturalGas(kBtu)',
    'GHGEmissionsIntensity',
    'ComplianceStatus'
]
available_drop = [col for col in columns_to_drop if col in df.columns]
print("Columns to drop found in dataset:", len(available_drop))
print(available_drop)
df = df.drop(columns=available_drop, errors='ignore')
print("Remaining columns after drop:", len(df.columns))

## 7. Save the cleaned dataset

The cleaned dataset is saved for the preparation and modeling notebooks.

In [ ]:
clean_path = "data/2016_Building_Energy_Benchmarking_ML.csv"
df.to_csv(clean_path, index=False)
print("Saved cleaned dataset to:", clean_path)

## 8. Detect energy and surface outliers

This section flags buildings that are extreme on both energy consumption and building surface.

In [ ]:
from scipy import stats

outlier_df = df.copy()
outlier_df = outlier_df[(outlier_df['SiteEnergyUse(kBtu)'] > 0) & (outlier_df['PropertyGFABuilding(s)'] > 0)]
outlier_df['z_score_energy'] = stats.zscore(outlier_df['SiteEnergyUse(kBtu)'])
q1 = outlier_df['PropertyGFABuilding(s)'].quantile(0.25)
q3 = outlier_df['PropertyGFABuilding(s)'].quantile(0.75)
iqr = q3 - q1
lower = q1 - 1.5 * iqr
upper = q3 + 1.5 * iqr
outlier_df['outlier_energy'] = outlier_df['z_score_energy'].abs() > 3
outlier_df['outlier_surface'] = (outlier_df['PropertyGFABuilding(s)'] < lower) | (outlier_df['PropertyGFABuilding(s)'] > upper)
final_outliers = outlier_df[outlier_df['outlier_energy'] & outlier_df['outlier_surface']].copy()
final_outliers['Surface_m2'] = final_outliers['PropertyGFABuilding(s)'] * 0.092903
final_outliers['Energy_kWh'] = final_outliers['SiteEnergyUse(kBtu)'] * 0.29307107
outlier_output = 'data/outliers_surface_energy.csv'
final_outliers.to_csv(outlier_output, index=False)
print("Saved combined energy+surface outliers to:", outlier_output)
print("Outliers detected:", len(final_outliers))
final_outliers[[ 'OSEBuildingID', 'SiteEnergyUse(kBtu)', 'Energy_kWh', 'PropertyGFABuilding(s)', 'Surface_m2', 'outlier_energy', 'outlier_surface']].head(20)